In [1]:
import numpy as np
import time
from collections import defaultdict, deque
import random

random.seed(42)

Parameters
----------
web_graph : dict[str, list[str]]
    Directed graph represented as {url: [outlink1, outlink2, ...]}
pagerank_scores : dict[str, float]
    Precomputed PageRank scores
crawl_allowed : dict[str, bool]
    Whether the URL permits crawling
k : int
    Number of URLs to return

Returns
-------
list[tuple[str, float]]
    Top-k URLs sorted by authority descending, with a small boost for trusted domains.

In [3]:
def prioritize_urls_with_heuristic(web_graph, pagerank_scores, crawl_allowed, k=5):
    
    trusted_markers = [".edu", ".gov", ".org"]
    ranked = []

    for url in web_graph:
        if not crawl_allowed.get(url, False):
            continue

        base_score = pagerank_scores.get(url, 0.0)

        trust_boost = 1.0
        if any(marker in url for marker in trusted_markers):
            trust_boost = 1.2

        final_score = base_score * trust_boost
        ranked.append((url, final_score))

    ranked.sort(key=lambda x: x[1], reverse=True)
    return ranked[:k]

In [20]:
web_graph = {
    "https://a.edu": ["https://b.com", "https://c.org"],
    "https://b.com": ["https://d.example"],
    "https://c.org": ["https://a.edu"],
    "https://d.example": ["https://c.org"],
    "https://e.example": ["https://b.com", "https://c.org"]
}

pagerank_scores = {
    "https://a.edu": 0.27,
    "https://b.com": 0.34,
    "https://c.org": 0.31,
    "https://d.example": 0.08,
    "https://e.example": 0.40
}

crawl_allowed = {
    "https://a.edu": True,
    "https://b.com": True,
    "https://c.org": True,
    "https://d.example": True,
    "https://e.example": False
}

In [22]:
top_pages = prioritize_urls_with_heuristic(web_graph, pagerank_scores, crawl_allowed, k=5)
print("Top URLs to crawl:")
for i, (url, score) in enumerate(top_pages, start=1):
    print(f"{i}. {url} -> {score:.4f}")

Top URLs to crawl:
1. https://c.org -> 0.3720
2. https://b.com -> 0.3400
3. https://a.edu -> 0.3240
4. https://d.example -> 0.0800
